# BFPC + ViT dual taxonomy (PMC9230753)

**Dos taxonomías:** ODIR-7 (sin H) · Cluster C (image k-means).

**Modelos por taxonomía:** BFPC `single` · BFPC `dual_view` · DINOv2 · ViT-ResidualAttention.

**Protocolo:** `val_loss` para checkpoint · `pos_weight` + umbrales con `recall_floor=0.80` · IAM paper en BFPC.

## Layout

```
MyDrive/retina_bfpc_vit_dual/
  code/bfpc_vit_dual_taxonomy_colab_bundle.zip   # solo código
  outputs/                                      # sync automático
```

**Datos RFMiD:** desde Kaggle (`USE_KAGGLE_DATA=True`) — no subas fotos a Drive.
Preferible: ejecutar todo en Kaggle GPU → `kaggle/README_KAGGLE.md`.


In [ ]:
from pathlib import Path
import os, sys, time, shutil, zipfile, subprocess

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/retina_bfpc_vit_dual")
DRIVE_CODE_ZIP = DRIVE_ROOT / "code" / "bfpc_vit_dual_taxonomy_colab_bundle.zip"
DRIVE_DATA = DRIVE_ROOT / "data" / "rfmid"  # solo si USE_KAGGLE_DATA=False
DRIVE_OUTPUTS = DRIVE_ROOT / "outputs"

PROJECT_ROOT = Path("/content/retina_multilabel")
EXP = PROJECT_ROOT / "experiments" / "bfpc_vit_dual_taxonomy"
PC = PROJECT_ROOT / "experiments" / "paper_compare_latent_clusters"

EPOCHS = 30
BATCH_BFPC = 16
BATCH_VIT = 8
NUM_WORKERS = 2
PATIENCE = 8
POS_WEIGHT_CAP = 20.0
RECALL_FLOOR = 0.80
SKIP_DONE = True
USE_KAGGLE_DATA = True  # recomendado: dataset andrewmvd/retinal-disease-classification
# Token API Kaggle (KGAT_...) — rellena al generar el notebook desde ~/.kaggle/kaggle_api_token
KAGGLE_API_TOKEN = __import__("os").environ.get("KAGGLE_API_TOKEN", "")  # set via Colab secret / env, never hardcode

for d in (DRIVE_ROOT / "code", DRIVE_OUTPUTS):
    d.mkdir(parents=True, exist_ok=True)
print("Drive OK", DRIVE_ROOT)


## 1b. Datos RFMiD desde Kaggle

Dataset: [andrewmvd/retinal-disease-classification](https://www.kaggle.com/datasets/andrewmvd/retinal-disease-classification) vía `kagglehub` + token `KGAT_...` (~7 GB a disco local Colab).

In [ ]:
def _find_rfmid(root: Path) -> Path:
    for labels in root.rglob("RFMiD_Training_Labels.csv"):
        for parent in [labels.parent, *labels.parents]:
            if (
                (parent / "Training_Set").is_dir()
                and (parent / "Evaluation_Set").is_dir()
                and (parent / "Test_Set").is_dir()
            ):
                return parent
    raise FileNotFoundError(f"RFMiD incompleto bajo {root}")

if USE_KAGGLE_DATA:
    token = (KAGGLE_API_TOKEN or os.environ.get("KAGGLE_API_TOKEN") or "").strip()
    assert token.startswith("KGAT_"), "Pon KAGGLE_API_TOKEN=KGAT_... en la celda 0"
    os.environ["KAGGLE_API_TOKEN"] = token
    %pip install -q kagglehub
    import kagglehub
    dl = Path(kagglehub.dataset_download("andrewmvd/retinal-disease-classification"))
    print("kagglehub download:", dl)
    DATA_ROOT = _find_rfmid(dl)
else:
    assert any(DRIVE_DATA.rglob("RFMiD_Training_Labels.csv")), f"Sube RFMiD a {DRIVE_DATA}"
    DATA_ROOT = _find_rfmid(DRIVE_DATA)

print("DATA_ROOT", DATA_ROOT)


## 1. Unzip + install

In [ ]:
assert DRIVE_CODE_ZIP.is_file(), f"Sube el ZIP a {DRIVE_CODE_ZIP}"

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True)

with zipfile.ZipFile(DRIVE_CODE_ZIP) as zf:
    zf.extractall(PROJECT_ROOT)

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    matches = list(PROJECT_ROOT.rglob("pyproject.toml"))
    assert matches, "pyproject.toml no está en el ZIP"
    nested = matches[0].parent
    if nested != PROJECT_ROOT:
        for child in nested.iterdir():
            dest = PROJECT_ROOT / child.name
            if not dest.exists():
                shutil.move(str(child), str(dest))

assert (PROJECT_ROOT / "pyproject.toml").is_file()
assert EXP.is_dir() and PC.is_dir()

%pip install -q --no-deps -e /content/retina_multilabel
%pip install -q "timm>=1.0" scikit-learn pandas matplotlib pillow tqdm huggingface_hub

os.environ["RETINA_DATA_ROOT"] = str(DATA_ROOT)
os.environ["TORCH_HOME"] = str(DRIVE_ROOT / "torch_cache")
Path(os.environ["TORCH_HOME"]).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PC))
sys.path.insert(0, str(EXP))

print("PROJECT_ROOT", PROJECT_ROOT)
print("RETINA_DATA_ROOT", os.environ["RETINA_DATA_ROOT"])


## 2. Helpers (sync Drive + run scripts)

In [ ]:
def run_script(rel_script: str, *args: str) -> None:
    script = EXP / "scripts" / rel_script
    cmd = [sys.executable, str(script), *args]
    print("\n>>>", " ".join(cmd), flush=True)
    t0 = time.time()
    subprocess.run(cmd, check=True, cwd=str(PROJECT_ROOT))
    print(f"<<< done in {time.time() - t0:.1f}s", flush=True)


def sync_outputs(label: str = "") -> None:
    src = EXP / "outputs"
    src.mkdir(parents=True, exist_ok=True)
    for path in src.rglob("*"):
        if path.is_dir() or "image_cache" in path.parts:
            continue
        rel = path.relative_to(src)
        out = DRIVE_OUTPUTS / rel
        out.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, out)
    print(f"[sync] {label} -> {DRIVE_OUTPUTS}")


def already_done(*markers) -> bool:
    return SKIP_DONE and all(Path(m).is_file() for m in markers)


# Restaurar outputs previos desde Drive
if DRIVE_OUTPUTS.exists() and any(DRIVE_OUTPUTS.iterdir()):
    for path in DRIVE_OUTPUTS.rglob("*"):
        if path.is_dir() or "image_cache" in path.parts:
            continue
        rel = path.relative_to(DRIVE_OUTPUTS)
        out = EXP / "outputs" / rel
        if out.exists() and SKIP_DONE:
            continue
        out.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, out)
    print("Restaurados outputs desde Drive (si habia).")


## 3. Datasets (ODIR-7 + Cluster C)

Si el ZIP ya trae `outputs/taxonomy_*`, solo verifica. Si faltan, regenera ODIR-7; Cluster C viene del bundle.

In [ ]:
odir7_map = EXP / "outputs" / "taxonomy_odir7" / "cluster_map.json"
c_map = EXP / "outputs" / "taxonomy_cluster_c" / "cluster_map.json"

need = not (odir7_map.is_file() and c_map.is_file())
if need or not SKIP_DONE:
    run_script("00_build_datasets.py")
else:
    print("SKIP datasets (ya en ZIP/Drive)")

assert odir7_map.is_file() and c_map.is_file()
import json
print("ODIR-7", json.loads(odir7_map.read_text())["cluster_names_ordered"])
print("Cluster C k=", len(json.loads(c_map.read_text())["cluster_names_ordered"]))
sync_outputs("datasets")


## 4. Entrenar (GPU)

Por cada taxonomía: BFPC single → BFPC dual → ViT DINOv2 → ViT residual.

Si OOM en ViT: baja `BATCH_VIT` a 4. Si DINOv2 falla por caché corrupta, limpia `torch_cache` en Drive.

In [ ]:
def train_all(taxonomy: str) -> None:
    jobs = [
        ("10_train_bfpc.py", f"bfpc_single_{taxonomy}",
         ["--taxonomy", taxonomy, "--mode", "single", "--batch-size", str(BATCH_BFPC)]),
        ("10_train_bfpc.py", f"bfpc_dual_view_{taxonomy}",
         ["--taxonomy", taxonomy, "--mode", "dual_view", "--batch-size", str(max(4, BATCH_BFPC // 2))]),
        ("11_train_vit.py", f"vit_dinov2_{taxonomy}",
         ["--taxonomy", taxonomy, "--batch-size", str(BATCH_VIT), "--img-size", "392"]),
        ("12_train_vit_residual.py", f"vit_residual_attn_{taxonomy}",
         ["--taxonomy", taxonomy, "--batch-size", str(BATCH_VIT), "--img-size", "392"]),
    ]
    common = [
        "--epochs", str(EPOCHS),
        "--num-workers", str(NUM_WORKERS),
        "--patience", str(PATIENCE),
        "--pos-weight-cap", str(POS_WEIGHT_CAP),
        "--recall-floor", str(RECALL_FLOOR),
    ]
    for script, run_name, extra in jobs:
        marker = EXP / "outputs" / "runs" / run_name / "run_summary.json"
        if already_done(marker):
            print(f"SKIP {run_name}")
            continue
        run_script(script, *common, "--run-name", run_name, *extra)
        sync_outputs(run_name)

print("=== ODIR-7 ===")
train_all("odir7")
print("=== Cluster C ===")
train_all("cluster_c")
sync_outputs("paper_final")


## 5. Comparar (recall / F1 / precision / val_loss)

In [ ]:
run_script("20_compare.py")
sync_outputs("compare")
cmp = EXP / "outputs" / "comparison" / "COMPARISON.md"
print(cmp.read_text()[:4000] if cmp.is_file() else "no COMPARISON.md yet")
print("Drive outputs:", DRIVE_OUTPUTS)
